## 1. 基于固定长度分块

In [2]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator="",       # 纯按长度切
    chunk_size=60,      # 依据实验与模型上限调整
    chunk_overlap=9,    # 15% 重叠
)

text = '''1.1 基于固定长度分块。
- 分块策略：按预设字符数 chunk_size 直接切分，不考虑文本结构。
- 优点：实现最简单、速度快、对任意文本通用。
- 缺点：容易破坏语义边界；块过大容易引入较多噪声，过小则会导致上下文不足。
- 适用场景：结构性弱的纯文本，或数据预处理初期的基线方案。'''
chunks = splitter.split_text(text)
print(chunks)

['1.1 基于固定长度分块。\n- 分块策略：按预设字符数 chunk_size 直接切分，不考虑文本结构。\n- 优点：实现', '。\n- 优点：实现最简单、速度快、对任意文本通用。\n- 缺点：容易破坏语义边界；块过大容易引入较多噪声，过小则会导致上下', '，过小则会导致上下文不足。\n- 适用场景：结构性弱的纯文本，或数据预处理初期的基线方案。']


## 2. 基于句子的分块

In [3]:
import re
def split_sentences_zh(text: str):
    # 在句末标点（。！？；）后面带可选引号的场景断句
    pattern = re.compile(r'([^。！？；]*[。！？；]+|[^。！？；]+$)')
    sentences = [m.group(0).strip() for m in pattern.finditer(text) if m.group(0).strip()]
    return sentences

def sentence_chunk(text: str, chunk_size=600, overlap=80):
    sents = split_sentences_zh(text)
    chunks, buf = [], ""
    for s in sents:
        if len(buf) + len(s) <= chunk_size:
            buf += s
        else:
            if buf:
                chunks.append(buf)
                # 简单重叠：从当前块尾部截取 overlap 字符与下一句拼接
            buf = (buf[-overlap:] if overlap > 0 and len(buf) > overlap else "") + s
    if buf:
        chunks.append(buf)
    return chunks

text = '''1.2 基于句子的分块
- 分块策略：先按句子切分，再将若干句子聚合成满足chunk_size的块；保证最基本的语义完整性。
- 优点：句子级完整性最好。对问句/答句映射友好。便于高质量引用。
- 缺点：中文分句需特别处理。仅句子级切分可能导致块过短，需后续聚合。
- 适用场景：法律法规、新闻、公告、FAQ 等以句子为主的文本。
- 中文分句注意事项：
  - 不要直接用 NLTK 英文 Punkt：无法识别中文标点，分句会失败或异常。
  - 可以直接使用以下内容进行分句：
    - 基于中文标点的正则：按“。！？；”等切分，保留引号与省略号等边界。
    - 使用支持中文的 NLP 库进行更精细的分句：
    - HanLP（推荐，工业级，支持繁多语言学特性）Stanza（清华/斯坦福合作，中文支持较好）spaCy + pkuseg 插件（或 zh-core-web-sm/med/lg 生态）
    '''

chunks = sentence_chunk(text, chunk_size=60, overlap=9)
print(chunks)

['1.2 基于句子的分块\n- 分块策略：先按句子切分，再将若干句子聚合成满足chunk_size的块；', 'k_size的块；保证最基本的语义完整性。- 优点：句子级完整性最好。对问句/答句映射友好。便于高质量引用。', '。便于高质量引用。- 缺点：中文分句需特别处理。仅句子级切分可能导致块过短，需后续聚合。', '过短，需后续聚合。- 适用场景：法律法规、新闻、公告、FAQ 等以句子为主的文本。', '以句子为主的文本。- 中文分句注意事项：\n  - 不要直接用 NLTK 英文 Punkt：无法识别中文标点，分句会失败或异常。', '分句会失败或异常。- 可以直接使用以下内容进行分句：\n    - 基于中文标点的正则：按“。！？；', '正则：按“。！？；”等切分，保留引号与省略号等边界。', '号与省略号等边界。- 使用支持中文的 NLP 库进行更精细的分句：\n    - HanLP（推荐，工业级，支持繁多语言学特性）Stanza（清华/斯坦福合作，中文支持较好）spaCy + pkuseg 插件（或 zh-core-web-sm/med/lg 生态）']


## 3. 基于递归字符分块

In [5]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
separators = [
    r"\n#{1,6}\s",                 # 标题
    r"\n\d+(?:\.\d+)*\s",          # 数字编号标题 1. / 2.3. 等
    "\n\n",                        # 段落
    "\n",                          # 行
    " ",                           # 空格
    "",                            # 兜底字符级
]
splitter = RecursiveCharacterTextSplitter(
    separators=separators,
    chunk_size=70,
    chunk_overlap=10,
    is_separator_regex=True,       # 告诉分割器上面包含正则
)

text = '''# 1.3 基于递归字符分块
## 分块策略：给定一组由“粗到细”的分隔符（如段落→换行→空格→字符），自上而下递归切分，在不超出 chunk_size 的前提下尽量保留自然语义边界。
## 优点：在“保持语义边界”和“控制块大小”之间取得稳健平衡，对大多数文本即插即用。
## 缺点：分隔符配置不当会导致块粒度失衡，极度格式化文本（表格/代码）效果一般。
## 适用场景：综合性语料、说明文档、报告、知识库条目。'''

chunks = splitter.split_text(text)
print(chunks)

['# 1.3 基于递归字符分块', '## 分块策略：给定一组由“粗到细”的分隔符（如段落→换行→空格→字符），自上而下递归切分，在不超出 chunk_size', '的前提下尽量保留自然语义边界。', '## 优点：在“保持语义边界”和“控制块大小”之间取得稳健平衡，对大多数文本即插即用。', '## 缺点：分隔符配置不当会导致块粒度失衡，极度格式化文本（表格/代码）效果一般。', '## 适用场景：综合性语料、说明文档、报告、知识库条目。']


## 4. 结构切分

In [7]:
"""演示按 Markdown 标题结构分块，并保留章节路径元数据。"""

import re
from typing import List, Dict

# 标题正则同时提取井号数量和标题文本，井号数量即标题层级。
heading_pat = re.compile(r'^(#{1,6})\s+(.*)$')
# 围栏代码块中的 # 可能是代码注释，不能当作 Markdown 标题处理。
fence_pat = re.compile(r'^```')


def split_markdown_structure(text, chunk_size=900, min_chunk=250, overlap_ratio=0.1) -> List[Dict]:
    """按照 Markdown 章节边界切分文本，并为每块附加章节元数据。

    chunk_size 是目标块的最大字符数，min_chunk 用于合并过短块。
    overlap_ratio 在当前示例中仅预留接口，尚未复制重叠文本。
    """
    # 第一步：逐行扫描文档，将正文归入对应的 Markdown 章节。
    lines = text.splitlines()
    sections = []
    in_code = False
    # 文档开头可能没有标题，因此先创建一个 level=0 的默认章节。
    current = {"level": 0, "title": "", "content": [], "path": []}

    # 栈中保存 (标题层级, 标题文本)，用于构建完整的章节路径。
    path_stack = []

    for ln in lines:
        # 每遇到一次 ``` 就切换代码块状态；代码块内不识别标题。
        if fence_pat.match(ln):
            in_code = not in_code
        m = heading_pat.match(ln) if not in_code else None
        if m:
            # 新标题意味着前一章节结束，先保存已经收集的正文。
            if current["content"]:
                sections.append(current)
            level = len(m.group(1))
            title = m.group(2).strip()

            # 弹出同级和更深层标题，只保留当前标题真正的父级路径。
            while path_stack and path_stack[-1][0] >= level:
                path_stack.pop()
            path_stack.append((level, title))
            # 例如 ["第一章", "1.1 小节"]，便于检索结果回溯原文位置。
            breadcrumbs = [t for _, t in path_stack]
            current = {"level": level, "title": title, "content": [], "path": breadcrumbs}
        else:
            current["content"].append(ln)

    # 扫描结束后，最后一个章节不会再遇到新标题，需要手动收尾。
    if current["content"]:
        sections.append(current)

    # 第二步：将章节正文转换为不超过 chunk_size 的初始文本块。
    chunks = []

    def emit_chunk(text_block: str, path: List[str], level: int):
        """统一封装文本和元数据，避免各分支重复组装字典。"""
        chunks.append({
            "text": text_block.strip(),
            "meta": {
                "section_title": path[-1] if path else "",
                "breadcrumbs": path,
                "section_level": level,
            }
        })

    for sec in sections:
        raw = "\n".join(sec["content"]).strip()
        if not raw:
            continue
        if len(raw) <= chunk_size:
            # 章节较短时整体输出，最大程度保留结构完整性。
            emit_chunk(raw, sec["path"], sec["level"])
        else:
            # 超长章节优先按空行拆成自然段，再逐段聚合到目标大小。
            paras = [p.strip() for p in  raw.split("\n\n") if p.strip()]
            buf = ""
            for p in paras:
                if len(buf) + len(p) + 2 <= chunk_size:
                    buf += (("\n\n" + p) if buf else p)
                else:
                    # 当前段放不下时提交已有缓冲区，再从当前段开始新块。
                    if buf:
                        emit_chunk(buf, sec["path"], sec["level"])
                    buf = p
            if buf:
                emit_chunk(buf, sec["path"], sec["level"])

    # 第三步：同一章节内的短块向前合并，减少过碎的检索单元。
    merged = []
    for ch in chunks:
        if not merged:
            merged.append(ch)
            continue
        if len(ch["text"]) < min_chunk and merged[-1]["meta"]["breadcrumbs"] == ch["meta"]["breadcrumbs"]:
            merged[-1]["text"] += "\n\n" + ch["text"]
        else:
            merged.append(ch)

    # overlap 当前仅计算未使用；实际项目可在此复制相邻块尾部，
    # 或在检索阶段通过相邻块扩展来补充上下文。
    overlap = int(chunk_size * overlap_ratio)
    for ch in merged:
        # 把最多五级章节路径写入正文前缀，让向量模型感知所属章节。
        bc = " > ".join(ch["meta"]["breadcrumbs"][-5:])
        prefix = f"[{bc}]\n" if bc else ""
        if prefix and not ch["text"].startswith(prefix):
            ch["text"] = prefix + ch["text"]
        # 当前示例选择在检索阶段做邻接聚合，因此这里不添加字符重叠。

    return merged




if __name__ == '__main__':

    text = open('data/visual_encoder.md', 'r', encoding='utf-8').read()

    chunks = split_markdown_structure(text, chunk_size=90, min_chunk=20)
    # print(chunks)
    for chunk in chunks:
        print(chunk['text'])
        print('=' * 100)

[视觉理解模型 > 一、视觉理解模型的框架原理 > 1. 整体架构组成]
现代视觉理解模型通常由以下几个关键模块构成：
- **视觉编码器（Visual Encoder）**
  负责将原始像素数据转化为高维语义特征表示。常用结构包括：
  - CNN（如 ResNet、EfficientNet）
  - Vision Transformer（ViT、Swin Transformer）
  - 多模态编码器（如 CLIP 的 image encoder）
- **语义/语言解码器（Semantic or Language Decoder，可选）**
  在需要生成文本的任务（如图像描述、VQA）中使用，例如：
  - Transformer Decoder
  - RNN/LSTM（早期方法）
[视觉理解模型 > 一、视觉理解模型的框架原理 > 1. 整体架构组成]
- **多模态融合模块（Multimodal Fusion Module）**
  在图文联合任务中（如 VQA、图文检索），用于对齐和融合视觉与语言特征。常见方式包括：
  - Cross-Attention（如 ViLBERT、LXMERT）
  - Early/Late Fusion
  - Co-Attention 或 Multimodal Transformer
- **任务头（Task Head）**
  针对具体下游任务设计的输出层，例如：
  - 分类头（FC + Softmax）
  - 检测头（如 Faster R-CNN 的 RPN + RoI Head）
  - 掩码预测头（用于分割）
[视觉理解模型 > 一、视觉理解模型的框架原理 > 2. 核心思想：表征学习 + 对齐 + 推理]
- **表征学习（Representation Learning）**：通过大规模数据预训练，学习通用视觉特征。
- **跨模态对齐（Alignment）**：在图文任务中，对齐视觉与语言空间（如 CLIP 的对比学习）。
- **上下文推理（Contextual Reasoning）**：利用注意力机制或图神经网络建模对象间关系（如 Relation Networks、Scene Graph）。

------
[视觉理解模型 > 二、视觉理解模型的构建方法 > 1. 数据准备]
- **标注数据

## 5. 对话切分

In [8]:
"""演示按轮次数和字符数切分多轮对话，并保留时间范围等元数据。"""

from typing import List, Dict

def chunk_dialogue(turns: List[Dict], max_turns=10, max_chars=900, overlap_turns=2):
    """使用滑动窗口把连续对话切成若干检索块。

    turns 格式：
    [{"speaker": "User", "text": "...", "ts_start": 123, "ts_end": 130}, ...]
    max_turns 和 max_chars 任一达到限制都会结束当前块；overlap_turns
    让相邻块共享若干轮对话，避免问句和答句恰好被切开。
    """
    chunks = []
    # i 是当前块的起始轮次索引。
    i = 0
    while i < len(turns):
        # j 持续向后试探，窗口范围使用左闭右开区间 [i, j)。
        j = i
        char_count = 0
        speakers = set()
        while j < len(turns):
            t = turns[j]
            uttr_len = len(t["text"])
            # 加入当前轮后若超过轮数或字符数限制，就结束窗口扩展。
            if (j - i + 1) > max_turns or (char_count + uttr_len) > max_chars:
                break
            char_count += uttr_len
            speakers.add(t["speaker"])
            j += 1

        if j > i:
            window = turns[i:j]
        elif i < len(turns):
            # 单轮自身超过 max_chars 时仍单独输出，避免该轮永远无法处理。
            window = [turns[i]]
        else:
            break
        # 将结构化轮次拼成适合向量化和检索的可读文本。
        text = "\n".join([f'{t["speaker"]}: {t["text"]}' for t in window])
        # 元数据用于按说话人过滤、定位原始轮次或回放对应时间片段。
        meta = {
            "speakers": list(speakers),
            "turns_range": (i, j - 1),
            "ts_start": window[0].get("ts_start"),
            "ts_end": window[-1].get("ts_end"),
        }
        chunks.append({"text": text, "meta": meta})

        # 最后一块处理完成后直接退出，无需再计算下一个窗口。
        if j >= len(turns):
            break
        # 下一块从当前块尾部向前回退 overlap_turns 轮，保留上下文连续性。
        next_start = i + len(window) - overlap_turns
        # overlap_turns 过大时仍保证至少前进一轮，防止死循环。
        i = max(next_start, i + 1)
    return chunks

turns = [
    {"speaker": "User",      "text": "今天天气真不错。",                "ts_start": 0,   "ts_end": 2},
    {"speaker": "Assistant", "text": "是啊，阳光正好。",                "ts_start": 3,   "ts_end": 5},
    {"speaker": "User",      "text": "要不要出去走走？",               "ts_start": 6,   "ts_end": 8},
    {"speaker": "Assistant", "text": "好主意，我正想出门透透气。",       "ts_start": 9,   "ts_end": 12},
    {"speaker": "User",      "text": "公园离这儿远吗？",               "ts_start": 13,  "ts_end": 15},
    {"speaker": "Assistant", "text": "不远，步行十分钟就到。",           "ts_start": 16,  "ts_end": 19},
    {"speaker": "User",      "text": "那我们现在就出发吧？",            "ts_start": 20,  "ts_end": 22},
    {"speaker": "Assistant", "text": "稍等，我拿一下水杯。",             "ts_start": 23,  "ts_end": 25},
    {"speaker": "User",      "text": "你带钥匙了吗？",                 "ts_start": 26,  "ts_end": 28},
    {"speaker": "Assistant", "text": "带了，在包里呢。",                 "ts_start": 29,  "ts_end": 31},
    {"speaker": "User",      "text": "路上小心点，人挺多的。",          "ts_start": 32,  "ts_end": 35},
    {"speaker": "Assistant", "text": "嗯，我会注意的。",                 "ts_start": 36,  "ts_end": 38},
    {"speaker": "User",      "text": "你看那只小狗多可爱！",            "ts_start": 39,  "ts_end": 42},
    {"speaker": "Assistant", "text": "真的！毛茸茸的，想摸摸它。",        "ts_start": 43,  "ts_end": 46},
    {"speaker": "User",      "text": "它主人好像在拍照。",              "ts_start": 47,  "ts_end": 49},
    {"speaker": "Assistant", "text": "我们别打扰他们了。",               "ts_start": 50,  "ts_end": 52},
    {"speaker": "User",      "text": "前面有家新开的咖啡馆。",          "ts_start": 53,  "ts_end": 56},
    {"speaker": "Assistant", "text": "待会儿回去可以试试。",             "ts_start": 57,  "ts_end": 59},
    {"speaker": "User",      "text": "记得你说喜欢抹茶拿铁？",          "ts_start": 60,  "ts_end": 63},
    {"speaker": "Assistant", "text": "对呀，希望他们家做得好喝。",        "ts_start": 64,  "ts_end": 67}
]

if __name__ == '__main__':
    '''
    max_turns_per_chunk：6–12 轮起步；语速快信息密度高可取 8–10。
    max_chars_per_chunk：600–1000 字；若存在长段独白，优先句级再切，不跨说话人。
    overlap_turns：1–2 轮；保证上一问下一答的连续性。
    '''
    chunks = chunk_dialogue(turns, max_turns=4, max_chars=90, overlap_turns=2)
    for chunk in chunks:
        print(chunk['text'])
        print('=' * 100)



User: 今天天气真不错。
Assistant: 是啊，阳光正好。
User: 要不要出去走走？
Assistant: 好主意，我正想出门透透气。
User: 要不要出去走走？
Assistant: 好主意，我正想出门透透气。
User: 公园离这儿远吗？
Assistant: 不远，步行十分钟就到。
User: 公园离这儿远吗？
Assistant: 不远，步行十分钟就到。
User: 那我们现在就出发吧？
Assistant: 稍等，我拿一下水杯。
User: 那我们现在就出发吧？
Assistant: 稍等，我拿一下水杯。
User: 你带钥匙了吗？
Assistant: 带了，在包里呢。
User: 你带钥匙了吗？
Assistant: 带了，在包里呢。
User: 路上小心点，人挺多的。
Assistant: 嗯，我会注意的。
User: 路上小心点，人挺多的。
Assistant: 嗯，我会注意的。
User: 你看那只小狗多可爱！
Assistant: 真的！毛茸茸的，想摸摸它。
User: 你看那只小狗多可爱！
Assistant: 真的！毛茸茸的，想摸摸它。
User: 它主人好像在拍照。
Assistant: 我们别打扰他们了。
User: 它主人好像在拍照。
Assistant: 我们别打扰他们了。
User: 前面有家新开的咖啡馆。
Assistant: 待会儿回去可以试试。
User: 前面有家新开的咖啡馆。
Assistant: 待会儿回去可以试试。
User: 记得你说喜欢抹茶拿铁？
Assistant: 对呀，希望他们家做得好喝。


## 语义切分

In [9]:
"""使用 Sentence Transformers 句向量检测主题变化并进行语义分块。"""

from typing import List, Dict, Tuple
import numpy as np
from sentence_transformers import SentenceTransformer
import re

# 安装
# pip install huggingface_hub[hf_xet]
# pip install sentence_transformers


def split_sentences_zh(text: str) -> List[str]:
    """按照常见中文句末标点切句，并保留句末标点。"""
    # 该正则适合教学示例；引号、省略号等复杂边界可交给 HanLP/Stanza。
    pattern = re.compile(r'([^。！？；]*[。！？；]+|[^。！？；]+$)')
    return [m.group(0).strip()  for m in pattern.finditer(text)  if m.group(0).strip()]


def rolling_mean(vecs: np.ndarray, i: int, w: int) -> np.ndarray:
    """计算以索引 i 为中心、左右最多各 w 个向量的局部均值。"""
    # max/min 用于裁剪首尾位置，避免窗口索引越界。
    s = max(0, i - w)
    e = min(len(vecs), i + w + 1)
    return vecs[s:e].mean(axis=0)

'''
语义分块:
分块策略：
    对⽂本先做句级切分，计算句⼦或短段的向量表示；
    当相邻语义的相似度显著下降（发⽣“语义突变”）时设为切分点。

参数调优说明（仅参考）
    阈值的含义：语义变化敏感度控制器，越低越容易切、越高越保守。
设定方式：
    绝对阈值：例如使用余弦相似度，若 sim < 0.75 则切分（需按语料校准）。
    相对阈值：对全篇的相似度/新颖度分布估计均值μ与标准差σ，使用 μ ± λσ 作为阈值，更稳健。
初始的配置建议（仅限于中文技术/说明文档）：
    窗口大小 window_size：2–4 句
    最小/最大块长：min_chunk_chars=300–400，max_chunk_chars=1000–1200
    阈值策略：novelty > μ + 0.8σ 或相似度 < μ - 0.8σ（先粗调后微调）
    overlap：10% 左右或按“附加上一句”做轻量轮次重叠
'''

def semantic_chunk(
        text: str,
        model_name: str = "BAAI/bge-m3",
        window_size: int = 2,
        min_chars: int = 3,
        max_chars: int = 110,
        lambda_std: float = 0.8,
        overlap_chars: int = 8,
) -> List[Dict]:
    """根据句子相对局部上下文的“新颖度”生成语义块。

    window_size 控制语义参照窗口，lambda_std 控制切分敏感度；
    min_chars 防止块过短，max_chars 为块大小提供硬上限。
    """
    # 第一步：先把整篇文本转换成最小语义单元——句子。
    sents = split_sentences_zh(text)
    print("sents:", sents)
    if not sents:
        return []

    # 第二步：加载嵌入模型并批量生成句向量。
    # normalize_embeddings=True 会把每个向量归一化，之后点积可视为余弦相似度。
    # 模型首次使用时会从 Hugging Face 下载，后续优先读取本地缓存。
    model = SentenceTransformer(model_name)
    emb = model.encode(sents, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    emb = np.asarray(emb)
    # print("emb.shape:", emb.shape)

    # 第三步：计算每句话相对于局部上下文的“新颖度”分数。
    novelties = []
    for i in range(len(sents)):
        # 以 i-1 为窗口中心生成参照向量；rolling_mean 是双侧窗口，
        # 因此参照可能包含该位置前后的邻近句子。
        ref = rolling_mean(emb, i- 1, window_size) if i > 0 else emb[0]  # 得到窗口
        # 均值向量不再保证单位长度，需要重新归一化。
        ref = ref / (np.linalg.norm(ref) + 1e-8)  # 归一化
        # 新颖度 = 1 - 相似度；分数越高，表示当前句与上下文差异越大。
        novelty = 1.0 - float(np.dot(emb[i], ref))
        novelties.append(novelty)
    novelties = np.array(novelties)
    # print("novelties:", novelties)  # 新颖度

    # 第四步：根据全文分数分布计算动态阈值 μ + λσ。
    # lambda_std 越大，阈值越高，只有更明显的主题变化才会触发切分。
    mu, sigma = float(novelties.mean()), float(novelties.std() + 1e-8)
    threshold = mu + lambda_std * sigma
    # print("threshold: ", threshold)  相对阈值

    chunks, buf, start_idx = [], "", 0

    def flush(end_idx: int):
        """提交缓冲区，并记录该块覆盖的句子索引范围。"""
        nonlocal buf, start_idx
        if buf.strip():
            chunks.append({
                "text": buf.strip(),
                "meta": {"start_sent": start_idx, "end_sent": end_idx - 1}
            })
        buf, start_idx = "", end_idx

    for i, s in enumerate(sents):
        # 第五步：逐句累积文本；即将超过硬上限时先提交已有内容。
        if len(buf) + len(s) > max_chars and len(buf) >= min_chars:
            flush(i)
            # 当前示例没有复制上一块尾部；此分支只处理比 overlap_chars
            # 更短的当前句，并将它直接放入新缓冲区。
            if overlap_chars > 0 and len(s) < overlap_chars:
                buf = s
                continue

        buf += s

        # 达到最小块长后，若当前句的新颖度超过阈值，就在此处切分。
        if len(buf) >= min_chars and novelties[i] > threshold:
            flush(i + 1)

    # 循环结束后提交不足 max_chars 的最后一个块。
    if buf:
        flush(len(sents))

    return chunks


text = '''清晨六点，城市已经苏醒。地铁站口涌出第一批通勤者，他们低头盯着手机屏幕，脚步匆忙而机械。高楼林立的街道上，玻璃幕墙反射着刺眼的阳光，空气中弥漫着汽车尾气与咖啡混合的味道。人们在狭窄的人行道上擦肩而过，却彼此视若无物——仿佛每个人都被装进了一个透明的茧里，既看得见世界，又无法真正触碰它。

这种快节奏的生活看似高效，实则暗藏代价。长期处于高压状态会削弱人的专注力，甚至引发焦虑与失眠。研究表明，连续数周暴露在噪音、拥挤和信息过载的环境中，大脑前额叶皮层的活跃度会显著下降，这直接影响决策能力和情绪调节。更令人担忧的是，许多人已将这种状态视为“正常”，甚至以“忙碌”为荣，误以为疲惫是成功的勋章。

然而，人类并非生来就适应钢筋水泥的丛林。我们的祖先曾在广袤的自然中狩猎、采集、休憩，感官与四季节律紧密相连。即便在现代社会，身体深处仍保留着对自然的渴望。这一点在心理学中被称为“亲生命性”（biophilia）——一种与生俱来的亲近自然的本能。

于是，越来越多的人开始逃离城市，哪怕只是短暂地。周末驱车两小时，走进一片未被过度开发的山林，成了都市人疗愈心灵的秘密仪式。脚踩松软的腐殖土，耳畔是溪流潺潺与鸟鸣交织的白噪音，鼻腔里充盈着松针与湿润泥土的清香——这些细微的感官刺激，竟能神奇地平复神经系统的躁动。有实验显示，仅在森林中漫步两小时，人体皮质醇（压力激素）水平平均下降15%以上。

更有趣的是，自然不仅修复情绪，还能激发创造力。当人远离电子设备与待办清单，大脑默认模式网络（Default Mode Network）会被激活，这是产生灵感、整合记忆与自我反思的关键区域。许多作家、艺术家都坦言，他们最重要的构思往往诞生于散步、观云或静坐湖边的时刻。自然不是逃避现实的避难所，而是重新连接内在智慧的桥梁。

当然，并非所有人都有条件频繁接触荒野。但城市本身也在悄然改变。近年来，“口袋公园”“垂直绿化”“社区农园”等微更新项目在各大城市兴起。哪怕是一面爬满常春藤的墙，或阳台上的几盆薄荷，也能带来微妙的心理慰藉。关键不在于自然的规模，而在于是否建立了真实的互动——亲手浇水、观察植物生长、感受微风拂过树叶的节奏。

值得警惕的是，我们正用技术模拟自然，却可能错失其本质。虚拟现实中的森林漫游、白噪音APP里的雨声，虽然能暂时缓解焦虑，但终究是感官的“代餐”。真正的疗愈来自全身心的沉浸：皮肤感受阳光的温度，双脚感知地面的起伏，眼睛追踪一只蝴蝶的飞行轨迹。这种多模态的体验无法被算法完全复刻。

归根结底，现代生活的困境不在于城市本身，而在于我们与自然节律的断裂。重建这种连接，并非要回归原始，而是学会在水泥缝隙中种下绿意，在日程表里留出“无所事事”的空白。或许，真正的自由不是拥有更多选择，而是有能力慢下来，听见一片叶子落地的声音。

当夜幕降临，城市再次亮起霓虹。但这一次，有人关掉屏幕，推开窗户，深深吸了一口带着凉意的晚风。他知道，明天依然要挤地铁、回邮件、赶截止日期，但此刻，他允许自己只是存在——像一棵树那样，安静而坚定。
'''

if __name__ == '__main__':
    chunks = semantic_chunk(text, window_size=2, min_chars=20, max_chars=1000)

    for chunk in chunks:
        print(chunk['text'])
        print('=' * 100)


ModuleNotFoundError: No module named 'sentence_transformers'

ollma

In [10]:
"""使用 Ollama 本地句向量检测主题变化并进行语义分块。"""

from typing import List, Dict, Tuple
import numpy as np
import re
from ollama import Client

# 安装
# pip install ollama


def split_sentences_zh(text: str) -> List[str]:
    """按照常见中文句末标点切句，并保留句末标点。"""
    # 该正则适合教学示例；引号、省略号等复杂边界可交给 HanLP/Stanza。
    pattern = re.compile(r'([^。！？；]*[。！？；]+|[^。！？；]+$)')
    return [m.group(0).strip()  for m in pattern.finditer(text)  if m.group(0).strip()]


def rolling_mean(vecs: np.ndarray, i: int, w: int) -> np.ndarray:
    """计算以索引 i 为中心、左右最多各 w 个向量的局部均值。"""
    # max/min 用于裁剪首尾位置，避免窗口索引越界。
    s = max(0, i - w)
    e = min(len(vecs), i + w + 1)
    return vecs[s:e].mean(axis=0)

'''
语义分块:
分块策略：
    对⽂本先做句级切分，计算句⼦或短段的向量表示；
    当相邻语义的相似度显著下降（发⽣“语义突变”）时设为切分点。

参数调优说明（仅参考）
    阈值的含义：语义变化敏感度控制器，越低越容易切、越高越保守。
设定方式：
    绝对阈值：例如使用余弦相似度，若 sim < 0.75 则切分（需按语料校准）。
    相对阈值：对全篇的相似度/新颖度分布估计均值μ与标准差σ，使用 μ ± λσ 作为阈值，更稳健。
初始的配置建议（仅限于中文技术/说明文档）：
    窗口大小 window_size：2–4 句
    最小/最大块长：min_chunk_chars=300–400，max_chunk_chars=1000–1200
    阈值策略：novelty > μ + 0.8σ 或相似度 < μ - 0.8σ（先粗调后微调）
    overlap：10% 左右或按“附加上一句”做轻量轮次重叠
'''
def semantic_chunk(
        text: str,
        model_name: str = "bge-m3",
        window_size: int = 2,
        min_chars: int = 3,
        max_chars: int = 110,
        lambda_std: float = 0.8,
        overlap_chars: int = 8,
) -> List[Dict]:
    """使用本地 Ollama 向量，根据句子新颖度生成语义块。

    window_size 控制语义参照窗口，lambda_std 控制切分敏感度；
    min_chars 防止块过短，max_chars 为块大小提供硬上限。
    """
    # 第一步：先把整篇文本转换成最小语义单元——句子。
    sents = split_sentences_zh(text)
    print("sents:", sents)
    if not sents:
        return []

    # 第二步：连接本机 Ollama 服务，批量请求所有句子的向量。
    # bge-m3 必须已经由当前 Ollama 实例加载或能够在模型列表中找到。
    client = Client("http://127.0.0.1:11434")
    emb = client.embed(model_name, sents, options={'num_gpu': 0})['embeddings']
    # 转成 NumPy 数组，方便后续求均值、范数和向量点积。
    emb = np.asarray(emb)
    print("emb.shape:", emb.shape)

    # 第三步：计算每句话相对于局部上下文的“新颖度”分数。
    novelties = []
    for i in range(len(sents)):
        # 以 i-1 为窗口中心生成参照向量；rolling_mean 是双侧窗口，
        # 因此参照可能包含该位置前后的邻近句子。
        ref = rolling_mean(emb, i- 1, window_size) if i > 0 else emb[0]  # 得到窗口
        # 均值向量需要重新归一化，1e-8 用于避免除零。
        ref = ref / (np.linalg.norm(ref) + 1e-8)  # 归一化
        # 新颖度 = 1 - 点积；这里假设模型返回的句向量已归一化。
        novelty = 1.0 - float(np.dot(emb[i], ref))
        novelties.append(novelty)
    novelties = np.array(novelties)
    # print("novelties:", novelties)  # 新颖度

    # 第四步：根据全文分数分布计算动态阈值 μ + λσ。
    # lambda_std 越大，阈值越高，只有更明显的主题变化才会触发切分。
    mu, sigma = float(novelties.mean()), float(novelties.std() + 1e-8)
    threshold = mu + lambda_std * sigma
    # print("threshold: ", threshold)  相对阈值

    chunks, buf, start_idx = [], "", 0

    def flush(end_idx: int):
        """提交缓冲区，并记录该块覆盖的句子索引范围。"""
        nonlocal buf, start_idx
        if buf.strip():
            chunks.append({
                "text": buf.strip(),
                "meta": {"start_sent": start_idx, "end_sent": end_idx - 1}
            })
        buf, start_idx = "", end_idx

    for i, s in enumerate(sents):
        # 第五步：逐句累积文本；即将超过硬上限时先提交已有内容。
        if len(buf) + len(s) > max_chars and len(buf) >= min_chars:
            flush(i)
            # 当前示例没有复制上一块尾部；此分支只处理比 overlap_chars
            # 更短的当前句，并将它直接放入新缓冲区。
            if overlap_chars > 0 and len(s) < overlap_chars:
                buf = s
                continue

        buf += s

        # 达到最小块长后，若当前句的新颖度超过阈值，就在此处切分。
        if len(buf) >= min_chars and novelties[i] > threshold:
            flush(i + 1)

    # 循环结束后提交不足 max_chars 的最后一个块。
    if buf:
        flush(len(sents))

    return chunks


text = '''清晨六点，城市已经苏醒。地铁站口涌出第一批通勤者，他们低头盯着手机屏幕，脚步匆忙而机械。高楼林立的街道上，玻璃幕墙反射着刺眼的阳光，空气中弥漫着汽车尾气与咖啡混合的味道。人们在狭窄的人行道上擦肩而过，却彼此视若无物——仿佛每个人都被装进了一个透明的茧里，既看得见世界，又无法真正触碰它。

这种快节奏的生活看似高效，实则暗藏代价。长期处于高压状态会削弱人的专注力，甚至引发焦虑与失眠。研究表明，连续数周暴露在噪音、拥挤和信息过载的环境中，大脑前额叶皮层的活跃度会显著下降，这直接影响决策能力和情绪调节。更令人担忧的是，许多人已将这种状态视为“正常”，甚至以“忙碌”为荣，误以为疲惫是成功的勋章。

然而，人类并非生来就适应钢筋水泥的丛林。我们的祖先曾在广袤的自然中狩猎、采集、休憩，感官与四季节律紧密相连。即便在现代社会，身体深处仍保留着对自然的渴望。这一点在心理学中被称为“亲生命性”（biophilia）——一种与生俱来的亲近自然的本能。

于是，越来越多的人开始逃离城市，哪怕只是短暂地。周末驱车两小时，走进一片未被过度开发的山林，成了都市人疗愈心灵的秘密仪式。脚踩松软的腐殖土，耳畔是溪流潺潺与鸟鸣交织的白噪音，鼻腔里充盈着松针与湿润泥土的清香——这些细微的感官刺激，竟能神奇地平复神经系统的躁动。有实验显示，仅在森林中漫步两小时，人体皮质醇（压力激素）水平平均下降15%以上。

更有趣的是，自然不仅修复情绪，还能激发创造力。当人远离电子设备与待办清单，大脑默认模式网络（Default Mode Network）会被激活，这是产生灵感、整合记忆与自我反思的关键区域。许多作家、艺术家都坦言，他们最重要的构思往往诞生于散步、观云或静坐湖边的时刻。自然不是逃避现实的避难所，而是重新连接内在智慧的桥梁。

当然，并非所有人都有条件频繁接触荒野。但城市本身也在悄然改变。近年来，“口袋公园”“垂直绿化”“社区农园”等微更新项目在各大城市兴起。哪怕是一面爬满常春藤的墙，或阳台上的几盆薄荷，也能带来微妙的心理慰藉。关键不在于自然的规模，而在于是否建立了真实的互动——亲手浇水、观察植物生长、感受微风拂过树叶的节奏。

值得警惕的是，我们正用技术模拟自然，却可能错失其本质。虚拟现实中的森林漫游、白噪音APP里的雨声，虽然能暂时缓解焦虑，但终究是感官的“代餐”。真正的疗愈来自全身心的沉浸：皮肤感受阳光的温度，双脚感知地面的起伏，眼睛追踪一只蝴蝶的飞行轨迹。这种多模态的体验无法被算法完全复刻。

归根结底，现代生活的困境不在于城市本身，而在于我们与自然节律的断裂。重建这种连接，并非要回归原始，而是学会在水泥缝隙中种下绿意，在日程表里留出“无所事事”的空白。或许，真正的自由不是拥有更多选择，而是有能力慢下来，听见一片叶子落地的声音。

当夜幕降临，城市再次亮起霓虹。但这一次，有人关掉屏幕，推开窗户，深深吸了一口带着凉意的晚风。他知道，明天依然要挤地铁、回邮件、赶截止日期，但此刻，他允许自己只是存在——像一棵树那样，安静而坚定。
'''

if __name__ == '__main__':
    chunks = semantic_chunk(text, window_size=2, min_chars=20, max_chars=1000)

    for chunk in chunks:
        print(chunk['text'])
        print('=' * 100)


ModuleNotFoundError: No module named 'ollama'